In [ ]:
# ========== 本练习：HuggingFace 上的 Llama 代码生成 + 文档润色 ==========
# 运行环境偏向 Google Colab（userdata 读 HF_TOKEN）
# HuggingFace LLAMA 代码生成器和验证器。


In [ ]:
# ========== 导入：Gradio UI + Transformers 本地推理 + Colab 密钥 ==========
# Gradio：快速搭建聊天界面
import gradio as gr
# PyTorch：张量运算与模型推理后端
import torch
# Transformers：分词器、因果语言模型、流式输出助手
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer
# Thread：把 generate 放到后台线程，主线程边收边 yield
from threading import Thread
# Hugging Face Hub 登录：拉取 gated 模型（如 Llama）需要 token
from huggingface_hub import login
# Google Colab userdata：从 Colab Secrets 读 HF_TOKEN，避免写进代码
from google.colab import userdata


In [ ]:
# ========== 登录 Hugging Face 并加载 Llama-3.2-1B-Instruct ==========
# 加载模型和分词器
# 模型 id（字符串勿改）：Meta 的指令微调小模型
model_name = "meta-llama/Llama-3.2-1B-Instruct"
# 提示正在下载/加载（首次可能较慢）
print(f"Loading {model_name}...")

# load_dotenv(覆盖=True)
# OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")
# 从 Colab Secrets 读取 HF_TOKEN（需事先配置）
hf_token = userdata.get('HF_TOKEN')
# 登录 HF Hub；add_to_git_credential=True 顺带写入 git 凭据缓存
login(hf_token, add_to_git_credential=True)

# 按模型名加载分词器（chat template 也在这里）
tokenizer = AutoTokenizer.from_pretrained(model_name)
# 因果 LM 常把 pad 设成 eos，避免生成时告警
tokenizer.pad_token = tokenizer.eos_token
# 加载权重：bfloat16 省显存；device_map="auto" 自动放到可用设备
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# 加载完成提示
print("Model loaded successfully!")


In [ ]:
# ========== 第二阶段：给已生成代码补 docstring（流式输出） ==========
def apply_docstrings(code):
    """
    Translator function to format the response.
    """
    # 文档助手 system prompt（英文原文勿改）
    sys_msg = """
      You are a technical assistant that documents Python code.
      Your task is below:
      - Add concise, clear, and informative docstrings to functions, classes, and modules.
      - Add inline comments only where they improve readability or clarify intent.
      - Do not modify the code logic or structure.
      - Give only the Python code and docstrings.
    """

    # user 消息：把上一阶段代码塞进「请加文档」请求
    usr_msg = f"""
    Add docstrings and comments to the following Python code.\n
    {code}
    """

    # 设置模型对话历史记录的格式
    # 设置模型对话：system + user
    messages = [{"role": "system", "content": sys_msg}, {"role": "user", "content": usr_msg}]

    # 应用聊天模板
    # 应用聊天模板：编成该模型期望的特殊 token 文本
    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # 对输入进行标记化chat_with_llama
    # 对输入分词，并搬到模型所在 device
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    # 设置流媒体
    # 流式解码器：跳过 prompt，只吐新生成 token
    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True
    )

    # 发电参数
    # 生成参数：max_new_tokens / temperature / top_p / 采样
    generation_kwargs = dict(
        inputs,
        streamer=streamer,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )

    # 在单独的线程中开始生成
    # 后台线程跑 model.generate，避免阻塞主线程读 streamer
    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    # 启动生成线程
    thread.start()

    # 流式传输响应
    # 累加流式片段，供 Gradio 实时刷新
    partial_response = ''
    # 从 streamer 逐段读取新文本
    for new_text in streamer:
        # 拼接已生成内容
        partial_response += new_text
        # yield 给上层（ChatInterface / chat_with_llama）
        yield partial_response


In [ ]:
# ========== 主聊天：先让 Llama 写代码，再交给 apply_docstrings 润色 ==========
def chat_with_llama(message, history):
    """
    Chat function that streams responses from the Llama model.
    Args:
        message: The user's current message
        history: List of [user_message, assistant_message] pairs
    Yields:
        Partial responses as they are generated
    """

    # 代码生成 system prompt（要求输出无注释的简洁 Python；原文勿改）
    sys_msg = """
      You are a expert python coder for a software company.
      You write python code for the specified problem.
      You never write comment in the code. Just provide raw and succinct python code.
      """
    # 设置模型对话历史记录的格式
    # 先放 system，再拼历史与当前用户消息
    messages = [{"role": "system", "content": sys_msg}]

    # 添加对话历史记录
    # Gradio history 是 [user, assistant] 对
    for user_msg, assistant_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})

    # 添加当前消息
    # 追加当前用户消息
    messages.append({"role": "user", "content": message})

    # 应用聊天模板
    # 应用聊天模板，得到模型可读的一整段 prompt 文本
    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # 对输入进行标记化chat_with_llama
    # 分词并搬到模型设备
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    # 生成响应
    # 推理阶段关闭梯度，省显存
    with torch.no_grad():
        # 非流式一次 generate：先得到「裸代码」
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )

    # 解码并返回响应
    # 只解码新生成部分（切开输入长度），去掉特殊 token
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    # 把裸代码交给文档助手流式补 docstring，并转发给 UI
    yield from apply_docstrings(response)


In [ ]:
# ========== 创建 Gradio ChatInterface 并启动（share 公网链接） ==========
# 创建 Gradio 界面
# 从 "org/model" 取出短模型名，用于标题展示
w_model = model_name.split('/')[-1]
# ChatInterface：把 chat_with_llama 接到聊天 UI
demo = gr.ChatInterface(
    # 回调：上面的两阶段 chat_with_llama
    fn=chat_with_llama,
    # UI 标题字符串保持原文
    title = f"🦙 {w_model} Chat",
    # UI 描述字符串保持原文
    description = f"Chat with Meta's {w_model} model with streaming responses",
    # 示例问题（英文原文，点击会填入输入框）
    examples=[
        "What is the capital of France?",
        "I want to travel to America",
        "What are some tips for learning a new language?"
    ],
    # Soft 主题
    theme=gr.themes.Soft()
)

# share=True 生成临时公网链接；debug=True 便于看报错
demo.launch(share=True, debug=True)



/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://eb2c5482d76228fa43.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
